In [1]:
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

os.makedirs('data/processed', exist_ok=True)
files = ['2020.xlsx', '2021.xlsx', '2022.xlsx', '2023.xlsx', '2024.xlsx', '2025.xlsx', '2026.xlsx']

records = []
for f in files:
    df_raw = pd.read_excel(f, sheet_name='Sheet')
    date_cols = [c for c in df_raw.columns if c not in ['No', 'Komoditas (Rp)']]
    for col in date_cols:
        records.append({
            'tanggal_str': col,
            'beras_raw': df_raw.loc[0, col],
            'beras_med1_raw': df_raw.loc[1, col]
        })

df = pd.DataFrame(records)
df['tanggal'] = pd.to_datetime(df['tanggal_str'].str.replace(' ', ''), format='%d/%m/%Y')
df = df.sort_values('tanggal').reset_index(drop=True)

def clean_price(x):
    s = str(x).replace(',', '').strip()
    return np.nan if s in ['-', '', 'nan', 'None'] else float(s)

df['harga_beras'] = df['beras_raw'].apply(clean_price).ffill()
df['harga_beras_medium1'] = df['beras_med1_raw'].apply(clean_price).ffill()
df = df[['tanggal', 'harga_beras', 'harga_beras_medium1']]

# Feature Engineering (Fokus Time Series & Siklus Kalender Umum)
for lag in [1, 3, 7, 14, 30]:
    df[f'lag_{lag}'] = df['harga_beras'].shift(lag)

df['lag_1_med1'] = df['harga_beras_medium1'].shift(1)
df['rolling_mean_7'] = df['harga_beras'].shift(1).rolling(window=7).mean()
df['rolling_std_7'] = df['harga_beras'].shift(1).rolling(window=7).std()
df['rolling_mean_30'] = df['harga_beras'].shift(1).rolling(window=30).mean()
df['bulan'] = df['tanggal'].dt.month
df['hari_dalam_minggu'] = df['tanggal'].dt.dayofweek
df['hari_dalam_bulan'] = df['tanggal'].dt.day

df_clean = df.dropna().reset_index(drop=True)

feature_cols = [
    'lag_1', 'lag_3', 'lag_7', 'lag_14', 'lag_30', 'lag_1_med1',
    'rolling_mean_7', 'rolling_std_7', 'rolling_mean_30',
    'bulan', 'hari_dalam_minggu', 'hari_dalam_bulan'
]
target_col = 'harga_beras'

# Chronological Split (70% Train, 15% Val, 15% Test)
n = len(df_clean)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df_clean.iloc[:train_end].copy()
val_df = df_clean.iloc[train_end:val_end].copy()
test_df = df_clean.iloc[val_end:].copy()

# Fit MinMaxScaler HANYA pada data training
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

scaler_X.fit(train_df[feature_cols])
scaler_y.fit(train_df[[target_col]])

# Simpan hasil ke folder data/processed/
train_df.to_csv('data/processed/train.csv', index=False)
val_df.to_csv('data/processed/val.csv', index=False)
test_df.to_csv('data/processed/test.csv', index=False)
with open('data/processed/scalers.pkl', 'wb') as f:
    pickle.dump({'scaler_X': scaler_X, 'scaler_y': scaler_y, 'features': feature_cols}, f)

print(f"Berhasil diperbarui dengan {len(feature_cols)} fitur! File siap di data/processed/.")

Berhasil diperbarui dengan 12 fitur! File siap di data/processed/.
